# EDA và tiền xử lý dữ liệu Wine Quality

Notebook thực hiện khảo sát và tiền xử lý bộ dữ liệu Wine Quality gồm rượu vang đỏ và trắng. Kết quả được lưu tại `output/wine_quality_clean.csv`.

**Bài toán:** phân loại đa lớp chất lượng rượu vang thành ba nhóm:
- **Thấp:** `quality <= 5`
- **Trung bình:** `quality == 6`
- **Cao:** `quality >= 7`

Nhãn số tương ứng: Thấp = 0, Trung bình = 1, Cao = 2. Cột `quality` gốc được giữ để phân tích, nhưng không được đưa vào tập đặc trưng khi huấn luyện mô hình cùng với `quality_group`/`label`.

## 1. Import thư viện, khai báo đường dẫn và đọc dữ liệu

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore", category=FutureWarning)

# Tìm thư mục dự án dựa trên các tệp dữ liệu mong đợi.
candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (p for p in candidate_roots
     if (p / "data" / "winequality-red.csv").is_file()
     and (p / "data" / "winequality-white.csv").is_file()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy data/winequality-red.csv và data/winequality-white.csv. "
        "Hãy đặt notebook trong thư mục notebooks/ của dự án hoặc cập nhật PROJECT_ROOT."
    )

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

red_path = DATA_DIR / "winequality-red.csv"
white_path = DATA_DIR / "winequality-white.csv"

red = pd.read_csv(red_path, sep=";")
white = pd.read_csv(white_path, sep=";")
red["wine_type"] = "red"
white["wine_type"] = "white"
df_raw = pd.concat([red, white], ignore_index=True)

print("Thư mục dự án:", PROJECT_ROOT)
print("Nguồn dữ liệu đỏ:", red_path)
print("Nguồn dữ liệu trắng:", white_path)
print("Kích thước rượu đỏ:", red.shape)
print("Kích thước rượu trắng:", white.shape)
print("Kích thước dữ liệu gộp:", df_raw.shape)
display(df_raw.head())

## 2. Khảo sát nguồn, quy mô, kiểu dữ liệu và từ điển dữ liệu

### Nguồn và điều kiện sử dụng

Nguồn dự kiến: **UCI Machine Learning Repository – Wine Quality** (dữ liệu rượu vang đỏ và trắng, nghiên cứu tại miền bắc Bồ Đào Nha). Trang nguồn: https://archive.ics.uci.edu/dataset/186/wine+quality

Bộ dữ liệu UCI được cung cấp theo giấy phép **CC BY 4.0**; khi sử dụng trong báo cáo, cần ghi nguồn và trích dẫn bài báo/bộ dữ liệu gốc theo hướng dẫn trên trang UCI. Nếu nhóm tải dữ liệu từ một nguồn khác, hãy ghi đúng nguồn thực tế đã sử dụng.

In [ ]:
print("Số mẫu:", len(df_raw))
print("Số cột:", df_raw.shape[1])
print("\nKiểu dữ liệu:")
display(df_raw.dtypes.astype(str).to_frame("data_type"))
print("\nThông tin tổng quan:")
df_raw.info()

In [ ]:
# Từ điển dữ liệu: mô tả các cột chuẩn của Wine Quality.
data_dictionary = pd.DataFrame([
    ("fixed acidity", "Độ axit cố định", "g/dm³", "Đặc trưng đầu vào"),
    ("volatile acidity", "Độ axit bay hơi", "g/dm³", "Đặc trưng đầu vào"),
    ("citric acid", "Axit citric", "g/dm³", "Đặc trưng đầu vào"),
    ("residual sugar", "Lượng đường dư", "g/dm³", "Đặc trưng đầu vào"),
    ("chlorides", "Nồng độ chloride", "g/dm³", "Đặc trưng đầu vào"),
    ("free sulfur dioxide", "Lượng sulfur dioxide tự do", "mg/dm³", "Đặc trưng đầu vào"),
    ("total sulfur dioxide", "Lượng sulfur dioxide tổng", "mg/dm³", "Đặc trưng đầu vào"),
    ("density", "Khối lượng riêng", "g/cm³", "Đặc trưng đầu vào"),
    ("pH", "Độ pH", "không đơn vị", "Đặc trưng đầu vào"),
    ("sulphates", "Nồng độ sulphates", "g/dm³", "Đặc trưng đầu vào"),
    ("alcohol", "Nồng độ cồn", "% thể tích", "Đặc trưng đầu vào"),
    ("quality", "Điểm chất lượng cảm quan gốc", "thang điểm", "Biến mục tiêu gốc"),
    ("wine_type", "Loại rượu: đỏ hoặc trắng", "red/white", "Đặc trưng phân loại bổ sung"),
], columns=["column", "meaning", "unit", "role"])
display(data_dictionary)

## 3. Thống kê mô tả

In [ ]:
numeric_cols = df_raw.select_dtypes(include=np.number).columns.tolist()
display(df_raw[numeric_cols].describe().T)

## 4. Phân tích phân bố đặc trưng và nhãn

In [ ]:
# Phân bố điểm chất lượng gốc
fig, ax = plt.subplots(figsize=(8, 4))
df_raw["quality"].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_title("Phân bố điểm chất lượng gốc (quality)")
ax.set_xlabel("Điểm chất lượng")
ax.set_ylabel("Số mẫu")
plt.tight_layout()
plt.show()

# Phân bố các đặc trưng số bằng histogram
df_raw[numeric_cols].hist(figsize=(16, 12), bins=30, edgecolor="black")
plt.suptitle("Phân bố các đặc trưng số", y=1.02)
plt.tight_layout()
plt.show()

# Phân bố theo loại rượu
fig, ax = plt.subplots(figsize=(6, 4))
df_raw["wine_type"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Số lượng mẫu theo loại rượu")
ax.set_xlabel("Loại rượu")
ax.set_ylabel("Số mẫu")
plt.tight_layout()
plt.show()

**Gợi ý nhận xét khi viết báo cáo:** mô tả đặc trưng có phân phối lệch, đặc trưng có khoảng biến thiên lớn, điểm chất lượng phổ biến và sự khác nhau về quy mô giữa hai loại rượu. Hãy dựa vào biểu đồ được sinh ra khi chạy notebook, không ghi nhận xét trái với kết quả thực tế.

## 5. Phân tích quan hệ đặc trưng – nhãn, kèm nhận xét

In [ ]:
# Tương quan giữa các biến số
corr = df_raw[numeric_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(12, 9))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
ax.set_title("Ma trận tương quan Pearson")
fig.colorbar(im, ax=ax, label="Hệ số tương quan")
plt.tight_layout()
plt.show()

# Tương quan với quality để sắp xếp các quan hệ tuyến tính nổi bật
quality_corr = corr["quality"].drop("quality").sort_values()
display(quality_corr.to_frame("correlation_with_quality"))

fig, ax = plt.subplots(figsize=(9, 5))
quality_corr.plot(kind="barh", ax=ax)
ax.set_title("Tương quan tuyến tính giữa đặc trưng và quality")
ax.set_xlabel("Hệ số tương quan Pearson")
plt.tight_layout()
plt.show()

# Boxplot các đặc trưng tiêu biểu theo điểm chất lượng
for col in ["alcohol", "volatile acidity", "sulphates", "density"]:
    fig, ax = plt.subplots(figsize=(7, 4))
    df_raw.boxplot(column=col, by="quality", ax=ax)
    ax.set_title(f"{col} theo điểm quality")
    ax.set_xlabel("Điểm quality")
    ax.set_ylabel(col)
    fig.suptitle("")
    plt.tight_layout()
    plt.show()

**Ghi ít nhất 3 nhận xét có ý nghĩa sau khi chạy:**
1. Nêu đặc trưng có tương quan dương/âm mạnh nhất với `quality` trong bảng kết quả và mức độ tương quan.
2. Nêu đặc trưng nào có phân phối thay đổi rõ giữa các mức `quality` trên boxplot.
3. Nêu một cặp đặc trưng có tương quan mạnh với nhau và giải thích khả năng thông tin trùng lặp/đa cộng tuyến.
4. Lưu ý: tương quan không chứng minh quan hệ nhân quả; quan hệ tuyến tính yếu cũng không có nghĩa đặc trưng hoàn toàn không hữu ích cho mô hình.

## 6. Kiểm tra dữ liệu thiếu, trùng, ngoại lai, sai định dạng

In [ ]:
# Kiểm tra dữ liệu thiếu
missing_report = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean() * 100).round(3)
}).sort_values("missing_count", ascending=False)
display(missing_report)

# Kiểm tra bản ghi trùng hoàn toàn
duplicate_count = int(df_raw.duplicated().sum())
print(f"Số bản ghi trùng hoàn toàn: {duplicate_count} / {len(df_raw)} "
      f"({duplicate_count / max(len(df_raw), 1) * 100:.2f}%)")

# Kiểm tra giá trị không phải số/sai định dạng ở các cột số
numeric_check = df_raw[numeric_cols].apply(pd.to_numeric, errors="coerce")
invalid_numeric = numeric_check.isna().sum() - df_raw[numeric_cols].isna().sum()
display(invalid_numeric.rename("invalid_non_numeric_count").to_frame())

# Phát hiện ngoại lai theo quy tắc IQR; chỉ thống kê, chưa tự động xóa
outlier_rows = []
for col in numeric_cols:
    s = pd.to_numeric(df_raw[col], errors="coerce").dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((s < lower) | (s > upper)).sum())
    outlier_rows.append((col, lower, upper, count, count / max(len(s), 1) * 100))
outlier_report = pd.DataFrame(outlier_rows, columns=[
    "feature", "IQR_lower_bound", "IQR_upper_bound", "outlier_count", "outlier_percent"
]).sort_values("outlier_count", ascending=False)
display(outlier_report)

**Lưu ý xử lý:** IQR đánh dấu điểm cần xem xét, không tự động chứng minh đó là lỗi. Với dữ liệu hóa lý rượu vang, cần xem xét miền giá trị và tài liệu nguồn trước khi xóa ngoại lai. Không loại ngoại lai chỉ để làm phân phối đẹp hơn.

## 7. Kiểm tra mất cân bằng nhãn và nguy cơ rò rỉ dữ liệu

In [ ]:
quality_counts = df_raw["quality"].value_counts().sort_index()
quality_pct = (quality_counts / quality_counts.sum() * 100).round(2)
display(pd.DataFrame({"count": quality_counts, "percent": quality_pct}))

fig, ax = plt.subplots(figsize=(7, 4))
quality_counts.plot(kind="bar", ax=ax)
ax.set_title("Phân bố nhãn gốc quality")
ax.set_xlabel("Quality")
ax.set_ylabel("Số mẫu")
plt.tight_layout()
plt.show()

# Kiểm tra các cột có thể gây rò rỉ sau khi tạo nhãn:
print("Cột mục tiêu gốc:", "quality")
print("Các cột dẫn xuất mục tiêu sẽ tạo ở bước sau: quality_group, label")
print("Khi huấn luyện, cần loại quality, quality_group và label khỏi X.")
print("Nếu có nhiều bản ghi cùng một mẫu vật/nguồn đo, cần cân nhắc chia theo nhóm để tránh cùng nhóm xuất hiện ở train và test.")

## 8. Tiền xử lý dữ liệu

In [ ]:
df_work = df_raw.copy()
rows_before = len(df_work)

# Ép kiểu số an toàn cho các cột số; giá trị không hợp lệ sẽ thành NaN và được báo cáo.
for col in numeric_cols:
    df_work[col] = pd.to_numeric(df_work[col], errors="coerce")

# Xóa dòng không có quality vì không thể tạo nhãn giám sát.
missing_quality_before = int(df_work["quality"].isna().sum())
df_work = df_work.dropna(subset=["quality"]).copy()

# Chỉ xóa bản ghi trùng hoàn toàn sau khi gộp dữ liệu đỏ/trắng.
duplicates_before_drop = int(df_work.duplicated().sum())
df_work = df_work.drop_duplicates().reset_index(drop=True)

# Với các đặc trưng còn thiếu, điền bằng trung vị của từng cột số.
# Việc điền ở đây chỉ tạo bộ dữ liệu EDA sạch; khi đánh giá mô hình,
# nên đặt imputer trong Pipeline và fit riêng trên tập huấn luyện.
feature_numeric_cols = [c for c in numeric_cols if c != "quality"]
missing_before_impute = df_work[feature_numeric_cols].isna().sum()
for col in feature_numeric_cols:
    if df_work[col].isna().any():
        median_value = df_work[col].median()
        df_work[col] = df_work[col].fillna(median_value)

# Loại các điểm quality ngoài thang điểm nguyên 0..10 nếu có.
valid_quality = (
    df_work["quality"].between(0, 10)
    & np.isclose(df_work["quality"], np.round(df_work["quality"]))
)
invalid_quality_count = int((~valid_quality).sum())
df_work = df_work.loc[valid_quality].copy()
df_work["quality"] = df_work["quality"].astype(int)

print("Số dòng ban đầu:", rows_before)
print("Dòng thiếu quality đã loại:", missing_quality_before)
print("Bản ghi trùng đã loại:", duplicates_before_drop)
print("Giá trị quality sai miền/không nguyên đã loại:", invalid_quality_count)
print("Số ô thiếu trước điền khuyết theo cột:")
display(missing_before_impute[missing_before_impute > 0].rename("missing_before_imputation").to_frame())
print("Kích thước sau tiền xử lý:", df_work.shape)
print("Số giá trị thiếu còn lại:", int(df_work.isna().sum().sum()))

## 9. Tạo nhãn chất lượng ba mức: Thấp – Trung bình – Cao

In [ ]:
def map_quality_group(q):
    if q <= 5:
        return "Thấp"
    elif q == 6:
        return "Trung bình"
    return "Cao"

label_map = {"Thấp": 0, "Trung bình": 1, "Cao": 2}
df_work["quality_group"] = df_work["quality"].map(map_quality_group)
df_work["label"] = df_work["quality_group"].map(label_map).astype("int64")

display(
    df_work[["quality", "quality_group", "label"]]
    .drop_duplicates()
    .sort_values("quality")
)
print("Phân bố nhãn sau khi tạo:")
display(pd.DataFrame({
    "count": df_work["quality_group"].value_counts(),
    "percent": (df_work["quality_group"].value_counts(normalize=True) * 100).round(2)
}))

## 10. So sánh dữ liệu trước và sau tiền xử lý

In [ ]:
comparison = pd.DataFrame({
    "before": {
        "rows": len(df_raw),
        "columns": df_raw.shape[1],
        "missing_cells": int(df_raw.isna().sum().sum()),
        "duplicate_rows": int(df_raw.duplicated().sum())
    },
    "after": {
        "rows": len(df_work),
        "columns": df_work.shape[1],
        "missing_cells": int(df_work.isna().sum().sum()),
        "duplicate_rows": int(df_work.duplicated().sum())
    }
})
comparison["change"] = comparison["after"] - comparison["before"]
display(comparison)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(["Trước", "Sau"], [len(df_raw), len(df_work)])
axes[0].set_title("So sánh số dòng")
axes[0].set_ylabel("Số dòng")
axes[1].bar(["Trước", "Sau"], [
    int(df_raw.duplicated().sum()),
    int(df_work.duplicated().sum())
])
axes[1].set_title("So sánh bản ghi trùng")
axes[1].set_ylabel("Số bản ghi trùng")
plt.tight_layout()
plt.show()

print("Phân bố nhãn trước xử lý (theo quality gốc):")
display(df_raw["quality"].value_counts().sort_index().rename("count").to_frame())
print("Phân bố nhãn ba mức sau xử lý:")
display(df_work["quality_group"].value_counts().reindex(["Thấp", "Trung bình", "Cao"]).rename("count").to_frame())

## 11. Lưu dữ liệu sạch vào `output/wine_quality_clean.csv`

In [ ]:
output_path = OUTPUT_DIR / "wine_quality_clean.csv"
df_work.to_csv(output_path, index=False, encoding="utf-8")
print("Đã lưu dữ liệu sạch tại:", output_path)
print("Kích thước tệp dữ liệu:", df_work.shape)
display(pd.read_csv(output_path).head())

## Ghi chú trước khi xây dựng mô hình

- Đây là bài toán **phân loại đa lớp**; đơn vị mẫu là một mẫu rượu.
- Khi tạo `X` và `y`, dùng `y = df["label"]`; loại khỏi `X` các cột `quality`, `quality_group`, `label`.
- Chỉ thực hiện mã hóa/chuẩn hóa và điền khuyết bằng `Pipeline` trên tập huấn luyện khi đánh giá mô hình để tránh rò rỉ dữ liệu.
- Có thể dùng `train_test_split(..., stratify=y)` vì dữ liệu không có trục thời gian được nêu trong bộ dữ liệu này. Đánh giá Macro-F1 cùng Accuracy và confusion matrix.
- Kết quả EDA và các nhận xét trong báo cáo phải dựa trên đầu ra thực tế sau khi chạy toàn bộ notebook.